# HW6
안세호 · 23013283

## 단계① 읽기와 나누기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

training : 803  test : 201


## 단계② model 셋의 test R²

In [2]:
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])
lin = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("linear regression       test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp  test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit) test R² :", round(tree.score(X_test, y_test), 3))

linear regression       test R² : 0.862
linear regression + qp  test R² : 1.0
decision tree (no limit) test R² : 0.994


## 단계③ 답

총액은 수량 × 단가인데, linear regression은 a × 수량 + b × 단가 + c 형태라 두 입력의 곱을 직접 표현하지 못해 test R²가 0.862에 그친다.  
수량 × 단가인 `qp`를 추가하면 그 열의 계수가 1, 나머지 계수와 절편이 0에 가까워져 총액 공식을 그대로 표현하므로 test R²가 1.0이 된다.

## 단계④ max_depth별 표

In [3]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## 단계⑤ cross-validation과 결정

In [4]:
for dep in [3, 5, 8, 12, None]:
    scores = cross_val_score(
        DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s}  mean {scores.mean():.3f}  std {scores.std():.3f}")

max_depth 3     mean 0.845  std 0.019
max_depth 5     mean 0.961  std 0.003
max_depth 8     mean 0.992  std 0.000
max_depth 12    mean 0.995  std 0.000


max_depth None  mean 0.994  std 0.001


max_depth는 8로 선택한다. 교차검증 평균 R²가 0.992로 높고, 잎 250개인 depth 8에서 잎 781개인 depth 12로 늘려도 평균 R²는 0.995로 0.003만 높아져 더 단순한 쪽을 선택했다.  
총액 = 수량 × 단가라는 규칙이 실제로 있으므로, 제한 없는 트리의 training R² 1.000과 test R² 0.994의 차이도 작다.

## 단계⑥ 모델 저장과 API

In [5]:
import joblib

order = pd.DataFrame([[40, 200000]], columns=["quantity", "unit_price"])
order2 = order.assign(qp=order["quantity"] * order["unit_price"])
tree8 = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)
print("linear + qp      →", int(round(lin2.predict(order2)[0])))
print("tree (depth 8)   →", int(round(tree8.predict(order)[0])))
print("quantity × price →", 40 * 200000)
joblib.dump(lin2, "sales_model.pkl")

linear + qp      → 8000000
tree (depth 8)   → 7362301
quantity × price → 8000000


['sales_model.pkl']

## 단계⑦ 답

API에는 linear regression + qp를 사용한다. 수량 40, 단가 200,000원일 때 예측한 8,000,000원이 실제 곱인 8,000,000원과 일치한다.  
depth 8 트리는 7,362,301원을 예측하므로, test R²가 1.0이며 총액 공식을 정확히 표현하는 linear regression + qp를 선택했다.  
서버에서도 `qp = quantity × unit_price`를 만들어 입력해야 하며, 실제 서비스에서는 공식이 있으므로 모델 없이 두 값을 곱하는 방법도 가능하다.